# Análisis Exploratorio de Datos — Sistema HVAC

**Proyecto:** HVAC AI System · Análisis de datos operativos de una empresa de climatización
**Autor:** Leonardo Orozco
**Fecha:** 2026-06-01

---

## Contexto

Este notebook explora los tres datasets fuente que alimentan el pipeline de análisis:

| # | Archivo | Contenido | Filas esperadas |
|---|---------|-----------|-----------------|
| 1 | `reporteMensual_FACTURAS.xlsx` | Facturas emitidas con fechas y montos | 396 |
| 2 | `CARTERA AL 11032026.xlsx` | Cuentas por cobrar y cotizaciones pendientes | 117 + 59 |
| 3 | `CONTROL DE INST. MINISPLIT 2026.xlsx` | Registro de instalaciones 2026 | — |

**Objetivo del EDA:** entender la estructura, calidad y distribuciones de cada dataset antes de construir modelos de scoring y forecast.

## 0. Configuración del entorno

In [ ]:
import pathlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Estilo visual ─────────────────────────────────────────────────────────────
# Se usa un estilo limpio con spines mínimos, adecuado para portfolio
sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

# Paleta de colores consistente en todo el notebook
PAL = {
    'azul':    '#2563EB',
    'naranja': '#F97316',
    'verde':   '#10B981',
    'rojo':    '#EF4444',
    'morado':  '#8B5CF6',
    'gris':    '#6B7280',
    'amarillo': '#F59E0B',
}
# Lista para seaborn (mismos colores en orden)
PALETTE = list(PAL.values())

# ── Rutas del proyecto ────────────────────────────────────────────────────────
# pathlib.Path('..') sube un nivel desde notebooks/ hasta la raíz del proyecto
ROOT = pathlib.Path('..').resolve()
RAW  = ROOT / 'data' / 'raw'
PROC = ROOT / 'data' / 'processed'

print('Entorno configurado correctamente')
print(f'  Raiz del proyecto: {ROOT}')
print(f'  Datos raw:         {RAW}')
print(f'  Datos procesados:  {PROC}')

---
## 1. Dataset: Facturas Mensuales

**Archivo:** `reporteMensual_FACTURAS.xlsx`

Este es el dataset central del proyecto. Contiene cada factura emitida por la empresa:
cuándo se emitió, a qué cliente, qué servicio se prestó, cuánto se cobró y —cuando aplica—
cuándo se recibió el pago.

Los campos clave para el análisis financiero son `Total` y `FECHA DE PAGO`.

In [ ]:
# ── Cargar y limpiar facturas ──────────────────────────────────────────────────

df_f = pd.read_excel(RAW / 'reporteMensual_FACTURAS.xlsx')

# El Excel tiene espacios extra en algunos nombres de columna (' Total ')
df_f.columns = df_f.columns.str.strip()

# Las 22 filas con Cliente=NaN son registros cancelados o separadores visuales
# Los descartamos antes del análisis
n_original = len(df_f)
df_f = df_f.dropna(subset=['Cliente']).copy()
print(f'Filas eliminadas (sin cliente): {n_original - len(df_f)}')

# Normalizar nombres de cliente: strip + uppercase
# Esto unifica variantes como "WALDOS " y "WALDOS"
df_f['Cliente'] = df_f['Cliente'].str.strip().str.upper()

# Parsear fechas — el Excel mezcla dos formatos:
#   Fecha:          "2025-03-12 00:00:00"  (ISO, generado por sistema)
#   FECHA DE PAGO:  "26/11/2025"           (dd/mm/yyyy, capturado manualmente)
df_f['Fecha']         = pd.to_datetime(df_f['Fecha'], errors='coerce')
df_f['FECHA DE PAGO'] = pd.to_datetime(df_f['FECHA DE PAGO'], dayfirst=True, errors='coerce')
df_f['Total']         = pd.to_numeric(df_f['Total'], errors='coerce')

# Columnas derivadas que usaremos en el análisis
df_f['mes_factura'] = df_f['Fecha'].dt.to_period('M')
df_f['mes_pago']    = df_f['FECHA DE PAGO'].dt.to_period('M')
df_f['dias_pago']   = (df_f['FECHA DE PAGO'] - df_f['Fecha']).dt.days.clip(lower=0)
df_f['pagada']      = df_f['FECHA DE PAGO'].notna()

print(f'Dataset listo: {len(df_f)} facturas')
df_f.head(4)

### 1.1 Estructura y calidad de datos

In [ ]:
print(f'Shape: {df_f.shape[0]} filas × {df_f.shape[1]} columnas')
print()

# Tipos de datos
print('── Tipos de datos ──────────────────────────────')
print(df_f.dtypes.to_string())
print()

# Valores nulos
nulos = df_f.isnull().sum()
pct   = (nulos / len(df_f) * 100).round(1)
resumen_nulos = pd.DataFrame({'Nulos': nulos, '%': pct})
print('── Valores nulos ───────────────────────────────')
print(resumen_nulos.to_string())
print()
print('NOTA: FECHA DE PAGO nula (41.2%) indica facturas aún no cobradas,')
print('      no errores de datos — es información de negocio.')

### 1.2 Estadísticas descriptivas

In [ ]:
print('══ KPIs principales ════════════════════════════════════════')
print(f"  Período:            {df_f['Fecha'].min().date()} → {df_f['Fecha'].max().date()}")
print(f"  Clientes únicos:    {df_f['Cliente'].nunique()}")
print(f"  Facturas pagadas:   {df_f['pagada'].sum()} ({df_f['pagada'].mean()*100:.1f}%)")
print(f"  Facturas pendientes:{(~df_f['pagada']).sum()} ({(~df_f['pagada']).mean()*100:.1f}%)")
print()
print(f"  Total facturado:    ${df_f['Total'].sum():>14,.2f}")
print(f"  Total cobrado:      ${df_f[df_f['pagada']]['Total'].sum():>14,.2f}")
print(f"  Total pendiente:    ${df_f[~df_f['pagada']]['Total'].sum():>14,.2f}")
print(f"  Tasa de cobro:      {df_f[df_f['pagada']]['Total'].sum() / df_f['Total'].sum() * 100:.1f}%")
print('════════════════════════════════════════════════════════════')
print()
print('── Estadísticas del campo Total (MXN) ──────────────────────')
print(df_f['Total'].describe().apply(lambda x: f'${x:>12,.2f}').to_string())

### 1.3 Gráfica 1 — Distribución de montos

La distribución de montos está **fuertemente sesgada a la derecha**: la mayoría de facturas
son de bajo valor (servicios de mantenimiento recurrente), mientras que pocas facturas de
alto valor (instalaciones o refacciones mayores) elevan la media significativamente.

Usamos escala logarítmica para visualizar toda la distribución correctamente.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribución de Montos — Facturas HVAC', fontsize=14, fontweight='bold', y=1.01)

# ── Subplot 1: Histograma en escala logarítmica ───────────────────────────────
ax = axes[0]
datos = df_f['Total'].dropna()
mediana = datos.median()
media   = datos.mean()

ax.hist(datos, bins=45, color=PAL['azul'], alpha=0.82, edgecolor='white', linewidth=0.4)
ax.axvline(mediana, color=PAL['naranja'], lw=2, ls='--', label=f'Mediana: ${mediana:,.0f}')
ax.axvline(media,   color=PAL['rojo'],   lw=2, ls=':',  label=f'Media:   ${media:,.0f}')
ax.set_xscale('log')
ax.set_xlabel('Monto MXN (escala log)')
ax.set_ylabel('Número de facturas')
ax.set_title('Histograma (escala log)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=9)

# ── Subplot 2: Boxplot pagadas vs pendientes ──────────────────────────────────
ax = axes[1]
pagadas    = df_f[df_f['pagada']]['Total'].dropna()
pendientes = df_f[~df_f['pagada']]['Total'].dropna()

bp = ax.boxplot(
    [pagadas, pendientes],
    patch_artist=True,
    widths=0.55,
    medianprops={'color': 'white', 'linewidth': 2.5},
    whiskerprops={'linewidth': 1.2},
    capprops={'linewidth': 1.2},
    flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.4},
)
bp['boxes'][0].set_facecolor(PAL['verde'])
bp['boxes'][1].set_facecolor(PAL['naranja'])

ax.set_xticklabels([
    f'Pagadas\n(n={len(pagadas)})',
    f'Pendientes\n(n={len(pendientes)})'
])
ax.set_yscale('log')
ax.set_ylabel('Monto MXN (escala log)')
ax.set_title('Boxplot: Pagadas vs Pendientes')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig(PROC / 'eda_01_distribucion_montos.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.4 Gráfica 2 — Ingresos cobrados por mes

Agrupa los cobros reales por mes de pago (`FECHA DE PAGO`). Esta es la vista de
**flujo de caja real**: el dinero que efectivamente ingresó cada mes.

Los meses en blanco (sin barra) son meses sin cobros registrados.

In [ ]:
# Agrupar ingresos por mes de pago
ingresos_mes = (
    df_f[df_f['pagada']]
    .groupby('mes_pago')['Total']
    .sum()
    .reset_index()
)
ingresos_mes.columns = ['mes', 'total']
ingresos_mes['mes_str'] = ingresos_mes['mes'].astype(str)

# Color diferencial: verde si supera la media, azul si no
media_mensual = ingresos_mes['total'].mean()
colores = [PAL['verde'] if v >= media_mensual else PAL['azul'] for v in ingresos_mes['total']]

fig, ax = plt.subplots(figsize=(14, 5))

bars = ax.bar(ingresos_mes['mes_str'], ingresos_mes['total'] / 1_000,
              color=colores, alpha=0.85, edgecolor='white', linewidth=0.5)

# Línea de media
ax.axhline(media_mensual / 1_000, color=PAL['naranja'], lw=1.8, ls='--',
           label=f'Media mensual: ${media_mensual/1_000:,.0f}k')

# Etiquetas de valor en las barras más altas
for bar, val in zip(bars, ingresos_mes['total']):
    if val >= media_mensual:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 8,
            f'${val/1_000:,.0f}k',
            ha='center', va='bottom', fontsize=7.5, color=PAL['gris']
        )

# Leyenda manual
patch_verde = mpatches.Patch(color=PAL['verde'], label=f'Sobre la media (>= ${media_mensual/1_000:,.0f}k)')
patch_azul  = mpatches.Patch(color=PAL['azul'],  label=f'Bajo la media')
ax.legend(handles=[patch_verde, patch_azul, ax.lines[0]], fontsize=9)

ax.set_title('Ingresos Cobrados por Mes (Flujo de Caja Real)', fontsize=14, fontweight='bold')
ax.set_xlabel('Mes de cobro')
ax.set_ylabel('Total cobrado (miles de MXN)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}k'))
plt.xticks(rotation=45, ha='right', fontsize=9)

plt.tight_layout()
plt.savefig(PROC / 'eda_02_ingresos_por_mes.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mes con mayor ingreso: {ingresos_mes.loc[ingresos_mes["total"].idxmax(), "mes_str"]}  '
      f'(${ingresos_mes["total"].max():,.0f})')
print(f'Mes con menor ingreso: {ingresos_mes.loc[ingresos_mes["total"].idxmin(), "mes_str"]}  '
      f'(${ingresos_mes["total"].min():,.0f})')

### 1.5 Gráfica 3 — Análisis por cliente

Dos dimensiones clave: el **volumen total facturado** (qué tan importante es
el cliente en términos de ingresos) y el **número de facturas** (frecuencia de
relación comercial).

**Hallazgo esperado:** TOYODA y WALDOS dominan el monto total, pero con perfiles
de pago muy distintos.

In [ ]:
# Resumen por cliente
resumen_cli = df_f.groupby('Cliente').agg(
    n_facturas=('Total', 'count'),
    monto_total=('Total', 'sum'),
    n_impagadas=('pagada', lambda x: (~x).sum()),
).sort_values('monto_total', ascending=False)

resumen_cli['pct_impago'] = (resumen_cli['n_impagadas'] / resumen_cli['n_facturas'] * 100).round(1)

# Top 10 por monto
top10 = resumen_cli.head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Análisis por Cliente', fontsize=14, fontweight='bold')

# ── Subplot 1: Monto total facturado ─────────────────────────────────────────
ax = axes[0]
colores_barra = [PAL['azul']] * len(top10)
ax.barh(top10.index[::-1], top10['monto_total'][::-1] / 1_000,
        color=colores_barra, alpha=0.85, edgecolor='white')
ax.set_xlabel('Monto total facturado (miles MXN)')
ax.set_title('Top 10 clientes por monto facturado')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}k'))
# Etiquetas de valor
for i, (idx, row) in enumerate(top10[::-1].iterrows()):
    ax.text(row['monto_total'] / 1_000 + 10, i,
            f"${row['monto_total']/1_000:,.0f}k", va='center', fontsize=8.5)

# ── Subplot 2: Número de facturas ─────────────────────────────────────────────
ax = axes[1]
# Ordenar por número de facturas para este subplot
top10_n = resumen_cli.sort_values('n_facturas', ascending=False).head(10)
colores_n = [PAL['naranja'] if row['pct_impago'] > 30 else PAL['azul']
             for _, row in top10_n.iterrows()]
ax.barh(top10_n.index[::-1], top10_n['n_facturas'][::-1],
        color=colores_n[::-1], alpha=0.85, edgecolor='white')
ax.set_xlabel('Número de facturas emitidas')
ax.set_title('Top 10 clientes por volumen de facturas\n(naranja = >30% impago)')
for i, (idx, row) in enumerate(top10_n[::-1].iterrows()):
    ax.text(row['n_facturas'] + 0.5, i,
            f"{row['n_facturas']}  ({row['pct_impago']}% imp.)", va='center', fontsize=8.5)

plt.tight_layout()
plt.savefig(PROC / 'eda_03_analisis_clientes.png', dpi=150, bbox_inches='tight')
plt.show()

print('Tabla completa de clientes:')
print(resumen_cli.to_string())

### 1.6 Gráfica 4 — Mapa de riesgo por cliente

Combina **valor total** (eje X) vs **porcentaje de impago** (eje Y) en un
scatter plot donde el tamaño del punto representa el número de facturas.

Cuadrante peligroso: alto valor + alto impago = riesgo financiero concentrado.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

sc = ax.scatter(
    resumen_cli['monto_total'] / 1_000,
    resumen_cli['pct_impago'],
    s=resumen_cli['n_facturas'] * 4,   # tamaño proporcional a # facturas
    c=resumen_cli['pct_impago'],        # color = nivel de riesgo
    cmap='RdYlGn_r',                    # rojo = alto impago, verde = bajo
    alpha=0.80,
    edgecolors='white',
    linewidths=0.7,
)

# Etiqueta con nombre del cliente
for idx, row in resumen_cli.iterrows():
    ax.annotate(
        idx,
        xy=(row['monto_total'] / 1_000, row['pct_impago']),
        xytext=(5, 4), textcoords='offset points',
        fontsize=7.5, color='#374151',
    )

# Líneas de referencia en percentil 50
med_monto  = resumen_cli['monto_total'].median() / 1_000
med_impago = resumen_cli['pct_impago'].median()
ax.axvline(med_monto,  color=PAL['gris'], lw=1, ls=':', alpha=0.7)
ax.axhline(med_impago, color=PAL['gris'], lw=1, ls=':', alpha=0.7)

# Anotaciones de cuadrantes
ax.text(ax.get_xlim()[1] * 0.98, ax.get_ylim()[1] * 0.95,
        'Alto valor\nAlto riesgo', ha='right', color=PAL['rojo'], fontsize=9, fontstyle='italic')
ax.text(ax.get_xlim()[1] * 0.98, 1,
        'Alto valor\nBajo riesgo', ha='right', color=PAL['verde'], fontsize=9, fontstyle='italic')

plt.colorbar(sc, ax=ax, label='% Facturas impagadas')
ax.set_xlabel('Monto total facturado (miles MXN)')
ax.set_ylabel('% Facturas impagadas')
ax.set_title('Mapa de Riesgo por Cliente\n(tamaño = número de facturas)', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}k'))

plt.tight_layout()
plt.savefig(PROC / 'eda_04_mapa_riesgo.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.7 Gráfica 5 — Velocidad de pago

¿Cuántos días tarda cada cliente en pagar sus facturas?
Esta métrica es clave para el modelo de scoring de clientes y para
anticipar tensiones de flujo de caja.

In [ ]:
# Solo facturas pagadas con días_pago > 0
dias = df_f[df_f['pagada'] & (df_f['dias_pago'] > 0)]['dias_pago']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Velocidad de Pago (días factura → cobro)', fontsize=14, fontweight='bold')

# ── Subplot 1: Histograma de días de pago ─────────────────────────────────────
ax = axes[0]
ax.hist(dias, bins=35, color=PAL['morado'], alpha=0.80, edgecolor='white', linewidth=0.4)
ax.axvline(dias.median(), color=PAL['naranja'], lw=2, ls='--',
           label=f'Mediana: {dias.median():.0f} días')
ax.axvline(dias.mean(),   color=PAL['rojo'],   lw=2, ls=':',
           label=f'Media:   {dias.mean():.0f} días')
ax.axvline(30, color=PAL['gris'], lw=1.2, ls=':', alpha=0.6, label='30 días (referencia)')
ax.set_xlabel('Días hasta el pago')
ax.set_ylabel('Número de facturas')
ax.set_title('Distribución de días de pago')
ax.legend(fontsize=9)

# ── Subplot 2: Mediana de días por cliente ────────────────────────────────────
ax = axes[1]
dias_cli = (
    df_f[df_f['pagada'] & (df_f['dias_pago'] > 0)]
    .groupby('Cliente')['dias_pago']
    .median()
    .sort_values()
)
colores_dias = [PAL['verde'] if d <= 30 else PAL['naranja'] if d <= 60 else PAL['rojo']
                for d in dias_cli]
ax.barh(dias_cli.index, dias_cli.values, color=colores_dias, alpha=0.82, edgecolor='white')
ax.axvline(30, color=PAL['gris'], lw=1.2, ls=':', alpha=0.6, label='30 días')
ax.axvline(60, color=PAL['naranja'], lw=1.2, ls=':', alpha=0.6, label='60 días')
ax.set_xlabel('Mediana de días hasta el pago')
ax.set_title('Días de pago mediana por cliente\n(verde ≤30d · naranja ≤60d · rojo >60d)')
ax.legend(fontsize=9)
for i, (cli, val) in enumerate(dias_cli.items()):
    ax.text(val + 0.5, i, f'{val:.0f}d', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(PROC / 'eda_05_dias_pago.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Facturas pagadas en <= 30 días: {(dias <= 30).sum()} ({(dias <= 30).mean()*100:.1f}%)')
print(f'Facturas pagadas en 31-60 días: {((dias > 30) & (dias <= 60)).sum()} ({((dias > 30) & (dias <= 60)).mean()*100:.1f}%)')
print(f'Facturas pagadas en > 60 días:  {(dias > 60).sum()} ({(dias > 60).mean()*100:.1f}%)')

---
## 2. Dataset: Cartera / Cuentas por Cobrar

**Archivo:** `CARTERA AL 11032026.xlsx`

Este archivo tiene **dos hojas** con distintos estados de cobranza:

- **OC FACTURADO:** Facturas ya emitidas con órdenes de compra, cuyo monto aún está
  pendiente de cobro según el sistema de cuentas por cobrar (`CURTRXAM`).
- **PTE OC 25-26:** Cotizaciones con orden de compra aún no facturadas — pipeline
  de ingresos futuros.

Los campos `CURTRXAM` y `ORCTRXAM1` son montos del sistema ERP del cliente; con
`CURTRXAM > 0` indica saldo pendiente de cobro.

In [ ]:
xl = pd.ExcelFile(RAW / 'CARTERA AL 11032026.xlsx')

# ── Hoja 1: OC Facturado ──────────────────────────────────────────────────────
# skiprows=2 porque las primeras 2 filas son título y espacio en blanco
df_oc = xl.parse(
    'OC FACTURADO', skiprows=2, usecols='A:E',
    names=['factura', 'oc', 'curtrxam', 'orctrxam1', 'fecha_calculo']
)
df_oc = df_oc.dropna(subset=['factura']).copy()
df_oc['curtrxam']      = pd.to_numeric(df_oc['curtrxam'],      errors='coerce')
df_oc['fecha_calculo'] = pd.to_datetime(df_oc['fecha_calculo'], errors='coerce')

# ── Hoja 2: Cotizaciones pendientes de OC ────────────────────────────────────
df_pte = xl.parse(
    'PTE OC 25-26', skiprows=3, usecols='B:E',
    names=['cot', 'suc', 'importe', 'concepto']
)
df_pte = df_pte.dropna(subset=['cot']).copy()
df_pte['importe'] = pd.to_numeric(df_pte['importe'], errors='coerce')

print('=== OC FACTURADO ===')
print(f'Shape: {df_oc.shape}')
print(df_oc.head(4).to_string())

print()
print('=== COTIZACIONES PTE OC 25-26 ===')
print(f'Shape: {df_pte.shape}')
print(df_pte.head(4).to_string())

### 2.1 Estructura y calidad de datos

In [ ]:
print('── OC FACTURADO ────────────────────────────────────────────')
print(f'Registros: {len(df_oc)}')
print(df_oc.dtypes.to_string())
print()
nulos_oc = df_oc.isnull().sum()
print('Nulos:')
print(pd.DataFrame({'Nulos': nulos_oc, '%': (nulos_oc/len(df_oc)*100).round(1)}).to_string())

print()
print('── COTIZACIONES PENDIENTES ─────────────────────────────────')
print(f'Registros: {len(df_pte)}')
print(df_pte.dtypes.to_string())

print()
print('── KPIs de Cartera ─────────────────────────────────────────')
cartera_total = df_oc['curtrxam'].sum()
cot_total     = df_pte['importe'].sum()
print(f'Saldo pendiente en cartera (OC):    ${cartera_total:>12,.2f}')
print(f'Pipeline de cotizaciones:           ${cot_total:>12,.2f}')
print(f'Total exposición:                   ${cartera_total + cot_total:>12,.2f}')
print()
print(f'Facturas en cartera: {len(df_oc)}')
print(f'Monto promedio por factura en cartera: ${df_oc["curtrxam"].mean():,.2f}')

### 2.2 Gráfica 6 — Distribución de saldos en cartera

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Análisis de Cartera — Cuentas por Cobrar', fontsize=14, fontweight='bold')

# ── Subplot 1: Distribución de montos en cartera ──────────────────────────────
ax = axes[0]
montos = df_oc['curtrxam'].dropna()
ax.hist(montos, bins=30, color=PAL['morado'], alpha=0.82, edgecolor='white', linewidth=0.4)
ax.axvline(montos.median(), color=PAL['naranja'], lw=2, ls='--',
           label=f'Mediana: ${montos.median():,.0f}')
ax.axvline(montos.mean(),   color=PAL['rojo'],   lw=2, ls=':',
           label=f'Media:   ${montos.mean():,.0f}')
ax.set_xlabel('Saldo pendiente (MXN)')
ax.set_ylabel('Número de facturas')
ax.set_title('Distribución de saldos en cartera')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=9)

# ── Subplot 2: Panorama financiero total ──────────────────────────────────────
ax = axes[1]
# Reconstruimos el panorama completo cruzando con facturas
total_facturado  = df_f['Total'].sum()
total_cobrado    = df_f[df_f['pagada']]['Total'].sum()
total_cartera_oc = df_oc['curtrxam'].sum()
total_cotizacion = df_pte['importe'].sum()

categorias = ['Facturado\ntotal', 'Cobrado', 'Pendiente\nen cartera', 'Pipeline\ncotizaciones']
valores    = [total_facturado, total_cobrado, total_cartera_oc, total_cotizacion]
colores_bar = [PAL['azul'], PAL['verde'], PAL['naranja'], PAL['morado']]

bars = ax.bar(categorias, [v / 1_000 for v in valores],
              color=colores_bar, alpha=0.85, edgecolor='white', width=0.6)

for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 15,
            f'${val/1_000:,.0f}k', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('Miles de MXN')
ax.set_title('Panorama financiero completo')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}k'))

plt.tight_layout()
plt.savefig(PROC / 'eda_06_cartera.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Dataset: Control de Instalaciones Minisplit

**Archivo:** `CONTROL DE INST. MINISPLIT 2026.xlsx`

In [ ]:
df_ms = pd.read_excel(RAW / 'CONTROL DE INST. MINISPLIT 2026.xlsx')

print('=== CONTROL DE INSTALACIONES MINISPLIT ===')
print(f'Shape: {df_ms.shape}')
print(f'Columnas: {list(df_ms.columns)}')
print()
print(df_ms.head())

### 3.1 Hallazgo: dataset vacío

El archivo existe y tiene la estructura de cabeceras definida (10 columnas), pero
**no contiene datos** — solo el encabezado de la tabla.

| Campo          | Descripción esperada                      |
|----------------|-------------------------------------------|
| ENERO          | Mes de registro                           |
| TECNICO        | Técnico que realizó la instalación        |
| CLIENTE        | Cliente                                   |
| REP #          | Número de representante / expediente      |
| DOMICILIO      | Dirección de la instalación               |
| TELEFONO       | Contacto                                  |
| TIPO DE TRABAJO| Tipo de instalación                       |
| PAGADO         | Monto pagado                              |
| RECIBE         | Quién recibió el pago                     |

**Implicación para el proyecto:** Este dataset está preparado para capturar
instalaciones de minisplit durante 2026 pero aún no ha sido llenado. Cuando
se popule, habilitará análisis de:
- Productividad por técnico
- Densidad geográfica de instalaciones
- Tasa de conversión cotización → instalación

> Este hallazgo es en sí mismo valioso en un EDA: identificar fuentes de
> datos planeadas pero aún vacías ayuda a priorizar el trabajo de captura.

---
## 4. Dataset: Conceptos Clasificados (Ground-Truth Labels)

**Archivo:** `data/processed/conceptos_clasificados.csv`

Este dataset no es una fuente operativa sino el **conjunto de etiquetas de entrenamiento**
para el clasificador NLP de conceptos de servicio. Contiene 73 conceptos únicos ya
clasificados en las 4 categorías del negocio.

In [ ]:
df_c = pd.read_csv(PROC / 'conceptos_clasificados.csv')

print(f'Shape: {df_c.shape}')
print(f'Columnas: {list(df_c.columns)}')
print()
print(df_c.head(5).to_string())
print()
print('── Distribución de categorías ───────────────────────────────')
print(df_c['categoria_modelo'].value_counts().to_string())

### 4.1 Gráfica 7 — Distribución de categorías de servicio

In [ ]:
conteo = df_c['categoria_modelo'].value_counts()

# Mapeo de etiquetas técnicas a nombres legibles
etiquetas_legibles = {
    'mantenimiento_preventivo': 'Mantenimiento\nPreventivo',
    'venta_refaccion':          'Venta de\nRefacciones',
    'instalacion_nueva':        'Instalación\nNueva',
    'mantenimiento_correctivo': 'Mantenimiento\nCorrectivo',
    'otro':                     'Otro',
}
colores_cat = [PAL['azul'], PAL['naranja'], PAL['verde'], PAL['morado'], PAL['gris']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribución de Categorías de Servicio\n(Conceptos Clasificados)',
             fontsize=14, fontweight='bold')

# ── Subplot 1: Donut chart ────────────────────────────────────────────────────
ax = axes[0]
wedges, texts, autotexts = ax.pie(
    conteo.values,
    labels=[etiquetas_legibles.get(k, k) for k in conteo.index],
    colors=colores_cat[:len(conteo)],
    autopct='%1.0f%%',
    startangle=90,
    pctdistance=0.78,
    wedgeprops={'width': 0.55, 'edgecolor': 'white', 'linewidth': 2},
)
for at in autotexts:
    at.set_fontsize(10)
    at.set_fontweight('bold')
ax.set_title(f'Distribución porcentual\n(Total: {len(df_c)} conceptos únicos)')

# ── Subplot 2: Bar chart con conteos ─────────────────────────────────────────
ax = axes[1]
ax.barh(
    [etiquetas_legibles.get(k, k).replace('\n', ' ') for k in conteo.index[::-1]],
    conteo.values[::-1],
    color=colores_cat[:len(conteo)][::-1],
    alpha=0.85,
    edgecolor='white',
)
for i, (cat, val) in enumerate(zip(conteo.index[::-1], conteo.values[::-1])):
    ax.text(val + 0.2, i, str(val), va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Número de conceptos etiquetados')
ax.set_title('Conceptos por categoría')
ax.set_xlim(0, conteo.max() + 4)

plt.tight_layout()
plt.savefig(PROC / 'eda_07_categorias_servicio.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 Gráfica 8 — Ingresos estimados por categoría (clasificación por palabras clave)

Los conceptos en `facturas` no coinciden exactamente con los de `conceptos_clasificados.csv`
porque el Excel captura texto libre con variantes (comas, fechas incluidas en el campo, etc.).

Para estimar la distribución de ingresos por categoría usamos un clasificador de **palabras clave**,
que será la línea base para el modelo NLP más avanzado:

| Categoría | Señales clave |
|-----------|---------------|
| `mantenimiento_preventivo` | "MANTENIMIENTO", "LIMPIEZA", "REVISION" |
| `mantenimiento_correctivo` | "REPARACION", "CORRECTIVO", "FUGA", "FALLA" |
| `instalacion_nueva`        | "INSTALACION", "INST.", "MINISPLIT" |
| `venta_refaccion`          | "COMPRESOR", "REFACCION", "PIEZA", "REFACCIONES" |

Este es exactamente el tipo de análisis que justifica construir el clasificador NLP.

In [ ]:
# Clasificador de palabras clave como línea base (baseline)
# Permite estimar la distribución de ingresos antes de tener el modelo NLP

REGLAS = {
    'mantenimiento_preventivo': [
        'MANTENIMIENTO', 'LIMPIEZA', 'REVISION', 'SERVICIO DE MANT',
        'PREVENTIVO', 'TRIMESTRAL', 'MENSUAL', 'SEMESTRAL',
    ],
    'mantenimiento_correctivo': [
        'CORRECTIVO', 'REPARACION', 'FUGA', 'FALLA', 'DIAGNOSTICO',
        'COMPLEMENTO', 'RECONSTRUIDO',
    ],
    'instalacion_nueva': [
        'INSTALACION', 'INST.', 'MINISPLIT', 'SPLIT', 'CHILLER',
        'EQUIPO NUEVO', 'SUMINISTRO E INST',
    ],
    'venta_refaccion': [
        'COMPRESOR', 'REFACCION', 'PIEZA', 'MOTOR', 'VALVULA',
        'CONDENSADOR', 'EVAPORADOR', 'CAPACITOR', 'GAS R',
    ],
}

def clasificar(concepto: str) -> str:
    # Devuelve la primera categoría cuyas palabras clave aparecen en el texto
    texto = str(concepto).upper()
    for categoria, palabras in REGLAS.items():
        if any(p in texto for p in palabras):
            return categoria
    return 'otro'

df_f['categoria'] = df_f['Concepto'].apply(clasificar)

# Resumen de cobertura
print('Cobertura del clasificador de palabras clave:')
print(df_f['categoria'].value_counts().to_string())
print(f"\nClasificadas (no 'otro'): {(df_f['categoria'] != 'otro').sum()} / {len(df_f)}")
print()

# Ingresos cobrados por categoría
ingresos_cat = (
    df_f[df_f['pagada']]
    .groupby('categoria')['Total']
    .agg(['sum', 'count', 'mean'])
    .sort_values('sum', ascending=False)
    .rename(columns={'sum': 'total', 'count': 'facturas', 'mean': 'ticket_medio'})
)
print('Ingresos cobrados por categoria:')
print(ingresos_cat.to_string())

# Gráfica
etiquetas_cat = {
    'mantenimiento_preventivo': 'Mtto.\nPreventivo',
    'venta_refaccion':          'Venta\nRefacciones',
    'instalacion_nueva':        'Instalacion\nNueva',
    'mantenimiento_correctivo': 'Mtto.\nCorrectivo',
    'otro':                     'Otro',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Ingresos Cobrados por Categoria de Servicio\n(Clasificacion por palabras clave — baseline)',
             fontsize=13, fontweight='bold')

colores_c = [colores_cat[i % len(colores_cat)] for i in range(len(ingresos_cat))]
etiq = [etiquetas_cat.get(k, k) for k in ingresos_cat.index]

ax = axes[0]
bars = ax.bar(etiq, ingresos_cat['total'] / 1_000,
              color=colores_c, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, ingresos_cat['total']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            f'${val/1_000:,.0f}k', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Total cobrado (miles MXN)')
ax.set_title('Ingresos totales por categoria')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}k'))
plt.setp(ax.get_xticklabels(), rotation=15, ha='right')

ax = axes[1]
bars2 = ax.bar(etiq, ingresos_cat['ticket_medio'] / 1_000,
               color=colores_c, alpha=0.85, edgecolor='white')
for bar, val in zip(bars2, ingresos_cat['ticket_medio']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'${val/1_000:,.1f}k', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Ticket promedio (miles MXN)')
ax.set_title('Ticket promedio por categoria')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}k'))
plt.setp(ax.get_xticklabels(), rotation=15, ha='right')

plt.tight_layout()
plt.savefig(PROC / 'eda_08_ingresos_categoria.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Resumen Ejecutivo

A continuación se consolidan los hallazgos principales del EDA en una vista unificada.

In [ ]:
print('╔══════════════════════════════════════════════════════════════════╗')
print('║              RESUMEN EJECUTIVO — EDA HVAC AI SYSTEM             ║')
print('╠══════════════════════════════════════════════════════════════════╣')
print('║                                                                  ║')
print('║  DATASET 1 — FACTURAS                                           ║')
print(f'║    Período analizado:   2025-01 → 2026-12                        ║')
print(f'║    Total facturado:     ${df_f["Total"].sum()/1_000_000:>5.2f}M MXN                           ║')
print(f'║    Total cobrado:       ${df_f[df_f["pagada"]]["Total"].sum()/1_000_000:>5.2f}M MXN                           ║')
print(f'║    Tasa de cobro:       {df_f[df_f["pagada"]]["Total"].sum()/df_f["Total"].sum()*100:>5.1f}%                                 ║')
print(f'║    Clientes activos:    {df_f["Cliente"].nunique():>3}                                     ║')
print(f'║    Cliente top:         TOYODA ($3.9M, 4% impago)                ║')
print(f'║    Riesgo principal:    WALDOS (223 facturas, 55% impago)         ║')
print('║                                                                  ║')
print('║  DATASET 2 — CARTERA                                            ║')
print(f'║    Saldo OC pendiente:  ${df_oc["curtrxam"].sum()/1_000:>7.1f}k MXN                      ║')
print(f'║    Pipeline cotizacion: ${df_pte["importe"].sum()/1_000:>7.1f}k MXN                      ║')
print('║                                                                  ║')
print('║  DATASET 3 — MINISPLIT                                          ║')
print('║    Estado: Plantilla vacía — sin datos capturados aún            ║')
print('║                                                                  ║')
print('║  DATASET 4 — CONCEPTOS CLASIFICADOS                             ║')
print(f'║    Etiquetas únicas:    {len(df_c):>3} conceptos en 5 categorías            ║')
print(f'║    Cobertura en facturas: {n_clasificadas/len(df_merged)*100:.0f}% de facturas clasificadas       ║')
print('║                                                                  ║')
print('╠══════════════════════════════════════════════════════════════════╣')
print('║  PRÓXIMOS PASOS                                                  ║')
print('║  1. Entrenar clasificador NLP con los 73 conceptos etiquetados   ║')
print('║  2. Expandir cobertura de categorías al 100% de facturas         ║')
print('║  3. Implementar scoring de clientes (riesgo + valor)             ║')
print('║  4. Conectar forecast de flujo de caja con el dashboard          ║')
print('║  5. Iniciar captura de datos de instalaciones (Dataset 3)        ║')
print('╚══════════════════════════════════════════════════════════════════╝')

---

*Notebook generado como parte del portafolio del proyecto HVAC AI System.*
*Para reproducir: ejecutar `python -X utf8 scripts/cargar_bd.py --limpiar` y luego correr este notebook.*